# BraTS 2024 - nnU-Net (2D) baseline

Prepares the raw BraTS-GLI volumes for **nnU-Net v2**, trains the `2d` configuration and compares it with the U-Net / Attention U-Net from `02_training.ipynb` on the **same patient split**.

nnU-Net does not use the preprocessed slice tensors. It reads full NIfTI volumes and the `2d` configuration trains on axial slices by itself.

Steps:
1. Paths and environment variables
2. Build `Dataset501_BraTSGLI` (hard links, no extra disk space) from `data_splits.json`
3. Plan and preprocess
4. Write `splits_final.json` with the project split
5. Train (run in a terminal, takes a long time)
6. Predict on the test set
7. Evaluate: global Dice per class, same definition as `Trainer` (restricted to the same z-range as the 2D slices)

`pip install nnunetv2` is required.

## 1. Paths and environment

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path

DATA_DIR = Path(r"E:\abronikowski\data")
RAW_CASES = DATA_DIR / "raw" / "training_data1_v2"
SPLITS_JSON = DATA_DIR / "data_splits.json"

NNUNET_BASE = Path(r"E:\abronikowski\nnunet")
os.environ["nnUNet_raw"] = str(NNUNET_BASE / "raw")
os.environ["nnUNet_preprocessed"] = str(NNUNET_BASE / "preprocessed")
os.environ["nnUNet_results"] = str(NNUNET_BASE / "results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

DATASET_ID = 501
DATASET_NAME = f"Dataset{DATASET_ID}_BraTSGLI"
DATASET_DIR = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
PREPROCESSED_DIR = Path(os.environ["nnUNet_preprocessed"]) / DATASET_NAME

# Channel order matches 02_training: T1, T1CE, T2, FLAIR
MODALITIES = ["t1n", "t1c", "t2w", "t2f"]

splits = json.loads(SPLITS_JSON.read_text())
print({k: len(v) for k, v in splits.items()})
print("Raw cases:", sum(1 for _ in RAW_CASES.iterdir()))

## 2. Build the nnU-Net dataset

`train` + `validate` go to `imagesTr` / `labelsTr` (the validation part is selected by `splits_final.json` below).
`test` goes to `imagesTs` / `labelsTs` and is never seen during training.

Files are **hard-linked** (same drive, no admin rights, no extra space); if linking fails, they are copied.

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def add_case(case: str, images_dir: Path, labels_dir: Path) -> None:
    src_dir = RAW_CASES / case
    for ch, mod in enumerate(MODALITIES):
        link_or_copy(src_dir / f"{case}-{mod}.nii.gz", images_dir / f"{case}_{ch:04d}.nii.gz")
    link_or_copy(src_dir / f"{case}-seg.nii.gz", labels_dir / f"{case}.nii.gz")

dirs = {n: DATASET_DIR / n for n in ("imagesTr", "labelsTr", "imagesTs", "labelsTs")}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

train_cases = splits["train"] + splits["validate"]
for case in train_cases:
    add_case(case, dirs["imagesTr"], dirs["labelsTr"])
for case in splits["test"]:
    add_case(case, dirs["imagesTs"], dirs["labelsTs"])

dataset_json = {
    "channel_names": {"0": "T1", "1": "T1CE", "2": "T2", "3": "FLAIR"},
    "labels": {"background": 0, "NCR": 1, "ED": 2, "ET": 3, "RC": 4},
    "numTraining": len(train_cases),
    "file_ending": ".nii.gz",
}
(DATASET_DIR / "dataset.json").write_text(json.dumps(dataset_json, indent=2))
print(f"{DATASET_NAME}: {len(train_cases)} training cases, {len(splits['test'])} test cases")

## 3. Plan and preprocess (2D only)

Takes a while (resampling and normalization of all training volumes).

In [ ]:
import importlib.util, sys

if importlib.util.find_spec("nnunetv2") is None:
    raise ImportError("nnunetv2 is not installed in this kernel's environment: run `pip install nnunetv2` and restart the kernel")

# Entry points called through the kernel's own interpreter, so it works even if the Scripts folder is not on PATH
ENTRY_POINTS = {
    "nnUNetv2_plan_and_preprocess": "nnunetv2.experiment_planning.plan_and_preprocess_entrypoints:plan_and_preprocess_entry",
    "nnUNetv2_train": "nnunetv2.run.run_training:run_training_entry",
    "nnUNetv2_predict": "nnunetv2.inference.predict_from_raw_data:predict_entry_point",
}

def run(cmd: list[str]) -> None:
    print(">", " ".join(cmd))
    module, func = ENTRY_POINTS[cmd[0]].split(":")
    code = f"import sys; from {module} import {func}; sys.argv[0] = {cmd[0]!r}; {func}()"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    # Stream output line by line so progress is visible while the command runs
    with subprocess.Popen([sys.executable, "-c", code, *cmd[1:]], env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


In [ ]:
# Already done once (1152 cases in nnUNetPlans_2d). Re-running overwrites the preprocessed data.
run(["nnUNetv2_plan_and_preprocess", "-d", str(DATASET_ID), "-c", "2d", "--verify_dataset_integrity"])

## 4. Use the project split

Replaces nnU-Net's default 5-fold split with a single fold: `train` for training, `validate` for validation. Train it as **fold 0**.

In [ ]:
splits_final = [{"train": splits["train"], "val": splits["validate"]}]
(PREPROCESSED_DIR / "splits_final.json").write_text(json.dumps(splits_final, indent=2))
print("Wrote", PREPROCESSED_DIR / "splits_final.json")

## 5. Training

Runs `nnUNetv2_train` on fold 0 and streams its output here. It takes many hours and blocks the kernel; closing VS Code or stopping the kernel stops the training. For an unattended run set `RUN_TRAINING = False` and paste the printed command into a terminal. The default trainer runs 1000 epochs of 250 iterations; `nnUNetTrainer_50epochs` is a sensible length for comparison with the U-Net.


In [ ]:
TRAINER = "nnUNetTrainer_50epochs"   # use "nnUNetTrainer" for the full 1000-epoch run
RUN_TRAINING = True                  # False: only print the command and run it in a terminal instead

cmd = ["nnUNetv2_train", str(DATASET_ID), "2d", "0", "-tr", TRAINER]
if RUN_TRAINING:
    run(cmd)   # blocks until training finishes; stopping the kernel stops the training
else:
    for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
        print(f'$env:{k} = "{os.environ[k]}"')
    print(" ".join(cmd))


## 6. Predict on the test set

In [ ]:
PRED_DIR = NNUNET_BASE / "predictions" / "test_2d"
PRED_DIR.mkdir(parents=True, exist_ok=True)

run(["nnUNetv2_predict", "-i", str(dirs["imagesTs"]), "-o", str(PRED_DIR),
     "-d", str(DATASET_ID), "-c", "2d", "-f", "0", "-tr", TRAINER,
     "-chk", "checkpoint_best.pth"])

## 7. Learning curves

Parsed from the nnU-Net training log (`training_log_*.txt` of the trainer used above; the newest log is used). `Pseudo dice` is nnU-Net's online validation Dice on patches (global over the validation set), so it is close to the Dice logged by `Trainer`. Saved to `checkpoints/learning_curves_nnunet.png`.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUT_DIR = PROJECT_ROOT / "checkpoints"
OUT_DIR.mkdir(exist_ok=True)

FOLD_DIR = Path(os.environ["nnUNet_results"]) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__2d" / "fold_0"
logs = sorted(FOLD_DIR.glob("training_log_*.txt"), key=lambda f: f.stat().st_mtime)
logs = [f for f in logs if "Epoch time" in f.read_text(errors="ignore")]
log_text = logs[-1].read_text(errors="ignore")
print("Using log:", logs[-1].name)

def series(name):
    return [float(v) for v in re.findall(rf"{name}\s+(-?\d+\.\d+)", log_text)]

train_loss, val_loss = series("train_loss"), series("val_loss")
pseudo = re.findall(r"Pseudo dice \[(.*?)\]", log_text)
pseudo = np.array([[float(x) for x in re.findall(r"float32\(([-\d.]+|nan)\)", row)] for row in pseudo])
mean_dice = np.nanmean(pseudo, axis=1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Train Loss", linewidth=2)
plt.plot(val_loss, label="Val Loss", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss During Training (nnU-Net 2d)")
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(mean_dice, label="Val Mean Dice (pseudo)", color="green", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Dice Score"); plt.title("Validation Mean Dice Score (nnU-Net 2d)")
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curves_nnunet.png", dpi=150)
print(f"Saved curve plot to: {OUT_DIR / 'learning_curves_nnunet.png'}")
plt.show()

## 8. Dice evaluation: validation vs test

Global Dice per class accumulated over all voxels (same as `dice_from_stats` in `brain_mri.models.metrics`), restricted to the z-range from `data/patients_z_boundaries.json`, so the numbers are comparable to `02_training.ipynb`. Validation predictions come from nnU-Net's own final validation run (`fold_0/validation`); test predictions from section 6. Note that nnU-Net's own `summary.json` reports Dice *per patient*, which is lower and not comparable.

In [ ]:
import nibabel as nib

z_bounds = json.loads((PROJECT_ROOT / "data" / "patients_z_boundaries.json").read_text())
CLASS_NAMES = {1: "NCR", 2: "ED", 3: "ET", 4: "RC"}

def load(path):
    return np.asanyarray(nib.load(path).dataobj).astype(np.uint8)

def global_dice(cases, pred_dir, gt_dir):
    inter = {c: 0.0 for c in CLASS_NAMES}
    total = {c: 0.0 for c in CLASS_NAMES}
    for case in cases:
        gt, pr = load(gt_dir / f"{case}.nii.gz"), load(pred_dir / f"{case}.nii.gz")
        if case in z_bounds:
            lo, hi = z_bounds[case]   # inclusive, as in get_slice_indices
            gt, pr = gt[..., lo:hi + 1], pr[..., lo:hi + 1]
        for c in CLASS_NAMES:
            g, p = gt == c, pr == c
            inter[c] += np.logical_and(g, p).sum()
            total[c] += g.sum() + p.sum()
    scores = {name: round(2 * inter[c] / total[c], 4) if total[c] else 1.0 for c, name in CLASS_NAMES.items()}
    scores["mean"] = round(sum(scores.values()) / len(CLASS_NAMES), 4)
    return scores

val_dice = global_dice(splits["validate"], FOLD_DIR / "validation", PREPROCESSED_DIR / "gt_segmentations")
test_dice = global_dice(splits["test"], PRED_DIR, dirs["labelsTs"])
print("Validation Dice:", val_dice)
print("Test Dice:      ", test_dice)

classes = list(CLASS_NAMES.values()) + ["mean"]
x = np.arange(len(classes))
width = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width / 2 - 0.01, [val_dice[c] for c in classes], width, label="Validation", color="#2a78d6")
test_bars = ax.bar(x + width / 2 + 0.01, [test_dice[c] for c in classes], width, label="Test", color="#eb6834")
ax.bar_label(test_bars, fmt="%.2f", padding=3, fontsize=9, color="#3d3d3a")

ax.set_xticks(x, classes)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Dice score")
ax.set_title("nnU-Net 2d - Dice per class: validation vs test (best checkpoint)")
ax.yaxis.grid(True, color="#e5e5e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.savefig(OUT_DIR / "test_dice_nnunet.png", dpi=150)
plt.show()

## 9. Example segmentations

Same layout and same random test patients (`seed=42`) as in `02_training.ipynb`: for each patient the axial slice with the largest tumor area. Last column: prediction as fill, ground truth as contour. Saved to `checkpoints/test_examples_nnunet.png`.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

N_PATIENTS = 4
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # NCR, ED, ET, RC
mask_cmap = ListedColormap(CLASS_COLORS)

rng = np.random.default_rng(42)
patients = rng.choice(sorted(splits["test"]), size=N_PATIENTS, replace=False)

def show_mask(ax, mask):
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=mask_cmap, vmin=1, vmax=4, alpha=0.55, interpolation="nearest")

def orient(a):
    return np.rot90(a)   # axial view; adjust (e.g. np.rot90(a, 3) or a.T) if it appears rotated compared to 02_training

fig, axes = plt.subplots(N_PATIENTS, 4, figsize=(14, 3.6 * N_PATIENTS))
for row, pid in enumerate(patients):
    gt, pr = load(dirs["labelsTs"] / f"{pid}.nii.gz"), load(PRED_DIR / f"{pid}.nii.gz")
    flair = np.asanyarray(nib.load(dirs["imagesTs"] / f"{pid}_0003.nii.gz").dataobj)
    z = int(np.argmax((gt > 0).sum(axis=(0, 1))))   # slice with the largest tumor area
    flair, gt, pr = orient(flair[..., z]), orient(gt[..., z]), orient(pr[..., z])
    wt_dice = 2 * ((gt > 0) & (pr > 0)).sum() / max((gt > 0).sum() + (pr > 0).sum(), 1)

    for ax in axes[row]:
        ax.imshow(flair, cmap="gray")
        ax.axis("off")
    show_mask(axes[row, 1], gt)
    show_mask(axes[row, 2], pr)
    show_mask(axes[row, 3], pr)
    for cls, color in zip(CLASS_NAMES, CLASS_COLORS):
        if (gt == cls).any():
            axes[row, 3].contour(gt == cls, levels=[0.5], colors=[color], linewidths=1.2)

    axes[row, 0].set_title(f"{pid}\nFLAIR", fontsize=10)
    axes[row, 1].set_title("Ground truth", fontsize=10)
    axes[row, 2].set_title("Prediction (nnU-Net 2d)", fontsize=10)
    axes[row, 3].set_title(f"Prediction (fill) + GT (contour)\nwhole-tumor Dice: {wt_dice:.2f}", fontsize=10)

handles = [Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES.values())]
handles.append(Line2D([], [], color="#3d3d3a", linewidth=1.2, label="GT contour"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=11)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig(OUT_DIR / "test_examples_nnunet.png", dpi=150)
plt.show()